In [1]:
import sys
sys.path.insert(1, '../../../scripts/')

import cobra
import gc
import pickle
from tqdm import tqdm
from core.model import ME_Model
from core.reaction import ME_Reaction
from macromolecules.RNA import mRNA
import pandas as pd
import numpy as np
import sympy
import multiprocessing
from macromolecules.RNA import pre_mRNA
import itertools
from macromolecules.macromolecule import Macromolecule
from macromolecules.RNA import rRNA
from macromolecules.protein import Protein
from utils import parameters as params
import copy


from expression import build_me_model

lp_path = '/data2/hratch/human_me/test_lp/'

No objective coefficients in model. Unclear what should be optimized


Identify the boundary value for feasibility for the numerator of c1

In [17]:
def replace_c1(x_add, mu_val = 1e-9):
    '''x_add is value to add to coupling coefficient c1'''
    
    with open('/data2/hratch/human_me/test_lp/' + 'dilution_coupled.pickle', 'rb') as handle:
        tme = pickle.load(handle) # model that failed due to c1 numerator, replace with diff numerator
    
    tme2 = ME_Model(id_or_model = '{}'.format(x_add), m_model = params.human_model)
    
    reactions = []
    for r_ in tqdm(tme.reactions): 
        cond1 = 'HGNC' in r_.id
        cond2 = ('TRANSLATION_ELONGATIONc' in r_.id) or ('co_TRANSLOC_IMPORTtr' in r_.id)
        cond3 = 'COMPLEX_FORMATION' not in r_.id

        r = r_.copy()
        if cond1 and cond2 and cond3:
            metabolites = {}

            m,c = [(m,c)for m,c in r.metabolites.items() if isinstance(m, mRNA)][0]
            frac = sympy.fraction(c)
            c = -(x_add/frac[1]) #replace mu with a specific value, independent of mu
#             c = ((frac[0] - x_add)/frac[1]) # add/substract a value to mu
            m.coupling_coefficient['mrna_dilution'] = -(c)
            metabolites[m] = c 

            r.add_metabolites(metabolites, combine = False)
        reactions.append(r)
        
    tme2.add_reactions(reactions)
    
    sln, stat, _ = tme2.solve_lp(mu_val = mu_val)
    
    store = {'x_add': x_add, 'mu_val': mu_val, 'stat': stat, 'sln': sln, 'model': tme2,  
             'infeasible_reactions': tme2.infeasible_reactions(mu_val = mu_val, sln = sln, stat = stat)}
    
    return store

def par(add, n_cores):
    pool = multiprocessing.Pool(processes = n_cores)
    try:
        stores = pool.map(replace_c1, add)
        pool.close()
        pool.join()
        gc.collect()
    except:
        pool.close()
        pool.join()
        gc.collect()
        print('failed')
    return stores

In [18]:
# stores = par(add = [0.05225, 0.0523], n_cores = 2)
stores = par(add=[1e-9, 0.07], n_cores = 2)

100%|██████████| 12650/12650 [03:20<00:00, 63.24it/s] 
../../../scripts/core/model.py:259 UserWarning: Solver is not initialized with ME_Model.intialize_solver, intializing with default parameters
../../../scripts/core/model.py:259 UserWarning: Solver is not initialized with ME_Model.intialize_solver, intializing with default parameters


Getting MINOS parameters...
Getting MINOS parameters...
Done in 236.017 seconds with status 1
Done in 287.895 seconds with status 0


../../../scripts/core/model.py:329 UserWarning: There is a discrepancy between the solver status and reactions that violate bound constraints


In [19]:
# # saveeee 
# for idx, x_add in list(enumerate(list(np.arange(1e-2, 1e-1, 10e-3)))):
#     stores[idx]['x_add'] = x_add

res = dict()
for store in stores:
    key = store.pop('x_add')
    res[key] = store
    


In [9]:
# for x_add, re in res.items():
#     if re['stat'].max()<1:
#         print(x_add)

between: 0.05225, 0.0523

___
# Explore

In [21]:
# res_ = {k:v for k,v in res.items() if k == 0.052300000000000006 or k == 0.052250000000000005}
mapper = dict(zip(sorted(res.keys()), ['infeasible', 'feasible']))
res_ = {mapper[k]:v for k,v in res.items()}

In [65]:
tol = pd.Series(list(res_['feasible']['infeasible_reactions'].values())).abs().max()

res_df = pd.DataFrame(columns = res_.keys(), index = [r.id for r in res_['infeasible']['model'].reactions])
for key in res_df.columns:
    for r_id in res_df.index:
        res_df.loc[r_id, key] = res_[key]['sln'][res_[key]['model'].reactions.index(r_id)]

res_df['difference'] = res_df.infeasible - res_df.feasible
res_df['rel_diff'] =  (res_df.infeasible - res_df.feasible)/res_df.feasible
res_df['abs_rel_diff'] = ((res_df.infeasible - res_df.feasible)/res_df.feasible).abs()
res_df.sort_values(by = 'abs_rel_diff', ascending = False, inplace = True)

summ = res_df[(res_df.difference != 0) & (res_df.infeasible.abs() > tol)]
summ.drop(columns = ['difference'], inplace = True)
summ = summ[summ.feasible != 0].sort_values(by = 'abs_rel_diff', ascending = False)

summ['r_id'] = summ.index
def prod(x):
    if x[1] < 0:
        return ';'.join([m.id for m in res_['feasible']['model'].reactions.get_by_id(x[0]).reactants if not isinstance(m, Macromolecule)])
    else:
        return ';'.join([m.id for m in res_['feasible']['model'].reactions.get_by_id(x[0]).products if not isinstance(m, Macromolecule)])
def reactant(x):
    if x[1] > 0:
        return ';'.join([m.id for m in res_['feasible']['model'].reactions.get_by_id(x[0]).reactants if not isinstance(m, Macromolecule)])
    else:
        return ';'.join([m.id for m in res_['feasible']['model'].reactions.get_by_id(x[0]).products if not isinstance(m, Macromolecule)])
    
summ['prod'] = summ[['r_id', 'infeasible']].apply(lambda x: prod(x), axis = 1).tolist()
summ['reactant'] = summ[['r_id', 'infeasible']].apply(lambda x: reactant(x), axis = 1).tolist()
summ.drop(columns = ['r_id'], inplace = True)

summ.head(7)

,infeasible,feasible,rel_diff,abs_rel_diff,prod,reactant
EX_lac_L_LPAREN_e_RPAREN_,-8.71586e-08,2.54895e-56,-3.4194e+48,3.4194e+48,lac_L_e,lac_L_b
EX_lac_L_b,-8.71586e-08,2.71964e-56,-3.20478e+48,3.20478e+48,lac_L_b,
r2129_F,8.71586e-08,-2.79957e-56,-3.11329e+48,3.11329e+48,bhb_e;lac_L_c,bhb_c;lac_L_e
EX_chol_b,-8.92739e-08,-3.62619e-56,2.46192e+48,2.46192e+48,chol_b,
EX_chol_LPAREN_e_RPAREN_,-8.92739e-08,-5.09789e-56,1.75119e+48,1.75119e+48,chol_e,chol_b
DHCR72r,8.13208e-11,-5.65809e-57,-1.43725e+46,1.43725e+46,nadp_r;chsterol_r,7dhchsterol_r;h_r;nadph_r
SRTNENT4tc_R,7.76232e-08,-1.10529e-53,-7.02291e+45,7.02291e+45,h_e;srtn_e,h_c;srtn_c


In [66]:
res_df.loc[[i for i in res_df.index if 'biomass' in i], :]

,infeasible,feasible,difference,rel_diff,abs_rel_diff
rRNA_biomass_to_biomass,3.05076e-10,2.28189e-14,3.05053e-10,13368.4,13368.4
tRNA_biomass_to_biomass,-5.27171e-23,1.01907e-22,-1.54624e-22,-1.51731,1.51731
other_rna_biomass_to_biomass,3.18191e-22,-6.71286e-22,9.89476e-22,-1.474,1.474
mRNA_biomass_to_biomass,8.32071e-18,3.6503e-10,-3.6503e-10,-1,1
protein_biomass_to_biomass,5.12924e-10,4.52947e-10,5.99766e-11,0.132414,0.132414
DNA_biomass_to_biomass,1.4e-11,1.4e-11,0,0,0
lipid_biomass_formation,1e-09,1e-09,0,0,0
carbohydrate_biomass_formation,1e-09,1e-09,0,0,0
DNA_biomass_formation,1e-09,1e-09,0,0,0
lipid_biomass_to_biomass,9.7e-11,9.7e-11,0,0,0


In [85]:
# reactions = [[r.id for r in list(m.reactions)] for m in res_['infeasible']['model'].metabolites if isinstance(m, pre_mRNA)]
# reactions = [item for sublist in reactions for item in sublist]
# rrna = res_df.loc[[i for i in reactions if i in res_df.index],:]
# rrna.sort_values(by = 'abs_rel_diff', ascending = False, inplace = True)
# rrna = rrna[rrna.infeasible.abs() > tol]
# rrna.head()

#reactions = [[r.id for r in list(m.reactions)] for m in res_['infeasible']['model'].metabolites if isinstance(m, rRNA)]
# reactions = [item for sublist in reactions for item in sublist]
reactions = [r.id for r in res_['infeasible']['model'].reactions if r.subsystem == 'Ribosome_Biogenesis']
rrna = res_df.loc[[i for i in reactions if i in res_df.index],:]
rrna.sort_values(by = 'abs_rel_diff', ascending = False, inplace = True)
rrna = rrna[rrna.infeasible.abs() > tol]
rrna.head()

,infeasible,feasible,difference,rel_diff,abs_rel_diff
ets_3_rRNA_DEGRADATIONn_0,7.54786e-14,9.81384e-18,7.54688e-14,7690.04,7690.04
FORMATION_RRNA_45s,7.54786e-14,9.81384e-18,7.54688e-14,7690.04,7690.04
TRANSCRIPTION_RRNA_47s,7.54786e-14,9.81384e-18,7.54688e-14,7690.04,7690.04
ets_5_frag1_rRNA_DEGRADATIONn_0,7.54786e-14,9.81384e-18,7.54688e-14,7690.04,7690.04
HGNC:10354_CYTOSOLIC_PROTEIN_FOLDING,3.44301e-16,9.81384e-18,3.34487e-16,34.0832,34.0832


In [106]:
[r.id for r in res_['feasible']['model'].reactions if 'TRANSLATION_ELONGATIONc' in r.id][-10]

'HGNC:30857_TRANSLATION_ELONGATIONc'

In [96]:
res_['feasible']['model'].reactions.get_by_id('HGNC:30857_TRANSLATION_ELONGATIONc').metabolites

{<Protein HGNC:30857_unfolded_protein_c at 0x7f1148e98cc0>: 1,
 <tRNA charged_generic_A_trna_c at 0x7f120cc4a588>: -25,
 <tRNA charged_generic_R_trna_c at 0x7f120ad121d0>: -20,
 <tRNA charged_generic_N_trna_c at 0x7f120ad12630>: -15,
 <tRNA charged_generic_D_trna_c at 0x7f1256bceac8>: -22,
 <tRNA charged_generic_C_trna_c at 0x7f125cda1ba8>: -8,
 <tRNA charged_generic_E_trna_c at 0x7f1240cf0518>: -14,
 <tRNA charged_generic_Q_trna_c at 0x7f123cda5b00>: -16,
 <tRNA charged_generic_G_trna_c at 0x7f1200cf2978>: -32,
 <tRNA charged_generic_H_trna_c at 0x7f1214dae550>: -10,
 <tRNA charged_generic_I_trna_c at 0x7f1210cf0710>: -17,
 <tRNA charged_generic_L_trna_c at 0x7f1284db7940>: -32,
 <tRNA charged_generic_K_trna_c at 0x7f1208afa588>: -19,
 <tRNA charged_generic_M_trna_c at 0x7f1250141358>: -7,
 <tRNA charged_generic_F_trna_c at 0x7f1200296d30>: -11,
 <tRNA charged_generic_P_trna_c at 0x7f120a24c908>: -17,
 <tRNA charged_generic_S_trna_c at 0x7f123cdcc6d8>: -31,
 <tRNA charged_generic_T_tr

In [97]:
res_['infeasible']['model'].reactions.get_by_id('HGNC:30857_TRANSLATION_ELONGATIONc').metabolites

{<Protein HGNC:30857_unfolded_protein_c at 0x7f11915b3710>: 1,
 <tRNA charged_generic_A_trna_c at 0x7f13aa977e10>: -25,
 <tRNA charged_generic_R_trna_c at 0x7f13cb163860>: -20,
 <tRNA charged_generic_N_trna_c at 0x7f12cd7d2908>: -15,
 <tRNA charged_generic_D_trna_c at 0x7f1212c01518>: -22,
 <tRNA charged_generic_C_trna_c at 0x7f12b3187e48>: -8,
 <tRNA charged_generic_E_trna_c at 0x7f121126ec88>: -14,
 <tRNA charged_generic_Q_trna_c at 0x7f12b3193518>: -16,
 <tRNA charged_generic_G_trna_c at 0x7f1241a28f60>: -32,
 <tRNA charged_generic_H_trna_c at 0x7f12105a5630>: -10,
 <tRNA charged_generic_I_trna_c at 0x7f12105a5668>: -17,
 <tRNA charged_generic_L_trna_c at 0x7f1261d9a710>: -32,
 <tRNA charged_generic_K_trna_c at 0x7f1261d9a550>: -19,
 <tRNA charged_generic_M_trna_c at 0x7f120b1a8898>: -7,
 <tRNA charged_generic_F_trna_c at 0x7f12108479b0>: -11,
 <tRNA charged_generic_P_trna_c at 0x7f12104d03c8>: -17,
 <tRNA charged_generic_S_trna_c at 0x7f12104d06a0>: -31,
 <tRNA charged_generic_T_tr

In [74]:
[m for m in mod.metabolites if 'HGNC:2501' in m.id if isinstance(m, Protein) and (() or ('COMPLEX_FORMATION'))]

[<Protein HGNC:25010_folded_pre_protein_m at 0x7f1209598908>,
 <Protein HGNC:2501_folded_protein_c at 0x7f12c0fb1dd8>,
 <Protein HGNC:2501_unfolded_protein_c at 0x7f11e14489e8>,
 <Protein HGNC:2501_folded_protein_c_polyub_protein_c at 0x7f11e14111d0>,
 <Protein HGNC:25010_unfolded_protein_c at 0x7f11632fd0b8>]

In [71]:
and m.coupling_coefficient is not None

In [75]:
mod.metabolites.get_by_id('HGNC:25010_folded_pre_protein_m')

Metabolite identifier,HGNC:25010_folded_pre_protein_m
Name,
Memory address,0x07f1209598908
Formula,C1261H2028N358O362S7
Compartment,m
In 3 reaction(s),"HGNC:25010_DEGRADATIONm, IMPORTtm_COMPLEX_FORMATIONm, HGNC:25010_IMPORTtm"


In [17]:
tol = pd.Series(list(res_['feasible']['infeasible_reactions'].values())).abs().max()

In [19]:
reactions = [[r.id for r in list(m.reactions)] for m in res_['infeasible']['model'].metabolites if isinstance(m, pre_mRNA)]
reactions = [item for sublist in reactions for item in sublist]
premrna = res_df.loc[[i for i in reactions if i in res_df.index],:]
premrna.sort_values(by = 'abs_rel_diff', ascending = False, inplace = True)
premrna = premrna[premrna.infeasible.abs() > tol]
premrna.head()

,infeasible,feasible,rel_diff,abs_rel_diff
HGNC:636_TRANSCRIPTION_PROCESSING,4.3474e-17,0,inf,inf
HGNC:636_TRANSCRIPTION_ELONGATION,4.3474e-17,1.1693e-56,3.71796e+39,3.71796e+39
HGNC:642_TRANSCRIPTION_ELONGATION,8.46814e-17,1.18959e-16,-0.288146,0.288146
HGNC:642_TRANSCRIPTION_PROCESSING,8.46814e-17,1.18959e-16,-0.288146,0.288146
HGNC:11071_TRANSCRIPTION_PROCESSING,2.9926e-18,3.06598e-18,-0.0239337,0.0239337


In [ ]:
# fail2 = res_df.loc[list(res_['infeasible']['infeasible_reactions'].keys()), :]
# tol = pd.Series(list(res_['feasible']['infeasible_reactions'].values())).abs().max()
# fail2 = fail2[fail2['infeasible'].abs() > tol]
# fail2.sort_values(by = 'abs_rel_diff', ascending = False, inplace = True)
# fail2.head(20)

In [20]:
r_id = 'GLYCTDle'
res_['infeasible']['model'].reactions.get_by_id(r_id).metabolites # HGNC 636

{<Metabolite glyc_e at 0x7f0810f42c18>: -1.0,
 <Metabolite glyc_c at 0x7f0811407780>: 1.0}

In [21]:
m_id = 'glyc_e'
test = res_df.loc[[r.id for r in res_['infeasible']['model'].metabolites.get_by_id(m_id).reactions if r.id in res_df.index],:]
test.sort_values(by = 'abs_rel_diff', ascending = False, inplace = True)
test

,infeasible,feasible,rel_diff,abs_rel_diff
H2OGLYAQPt_R_0,1.60182e-07,0,inf,inf
GLYCTDle,1.17649e-07,-4.25323e-08,-3.76611,3.76611


In [22]:
r_ids = ['H2OGLYAQPt_R_0', 'GLYCTDle']
add = list()
for r_id in r_ids:
    add += [m.id for m in res_['infeasible']['model'].reactions.get_by_id(r_id).metabolites]
add = sorted(set(add))

In [270]:
r_ids = ['H2OGLYAQPt_R_0', 'GLYCTDle']
add = list()
for r_id in r_ids:
    add += [m.id for m in res_['infeasible']['model'].reactions.get_by_id(r_id).metabolites]
add = sorted(set(add))


tm = ME_Model(id_or_model = 'tm', m_model = params.human_model)
reactions = [r.copy() for r in res_['infeasible']['model'].reactions]

print('add metabolites')
for m_id in add:
    r = cobra.Reaction('SK_' + m_id)
    r.bounds = (-1000,1000)
    r.add_metabolites({res_['infeasible']['model'].metabolites.get_by_id(m_id).copy(): -1})
    reactions.append(r)
    
tm.add_reactions(reactions)

print('solve')
sln, stat, _ = tm.solve_lp(mu_val = 1e-9)

add metabolites
solve


../../../scripts/core/model.py:259 UserWarning: Solver is not initialized with ME_Model.intialize_solver, intializing with default parameters


Getting MINOS parameters...
Done in 235.84 seconds with status 0



Notes: 

comparing infeasible and feasible values at the boundary, where numerator of c1 must be a minimal value for feasibility

this range is b/w 0.05225, 0.0523

Observations:

1) premrna biomass flux is larger in infeasible version-->
2) flux for generating premrna for HGNC:636 is very high-->
3) HGNC:636 is involved in catalysis reactions for h2o and glycine metabolites in e/c compartments
4) add all metabolites from the reactions it is involved in as sinks makes model feasible, but unsure yet whether this is just due to adding the protein, or the metabolites
5) looks like model is trying to generate more h2o_c, adding this as a sink makes it feasible

# Add sinks

In [2]:
lp_path = '/data2/hratch/human_me/test_lp/'

def add_sink(metabolite_ids, mu_val = 1e-9, model = None):
    '''metabolite ids is a list of metabolite ids to add as sinks and test model feasibility'''
    
    if model is None:
        with open('/data2/hratch/human_me/test_lp/' + 'infeasible_boundary.pickle', 'rb') as handle:
            model = pickle.load(handle) # boundary infeasible
    model.initialize_solver(solver_type='qminos', precision='quad')
    
    metabolite_ids = sorted(set(metabolite_ids))
    reactions = [r.copy() for r in tqdm(model.reactions)]
    for m_id in metabolite_ids:
        r = cobra.Reaction('SK_' + m_id)
        r.bounds = (-1000,1000)
        r.add_metabolites({model.metabolites.get_by_id(m_id).copy(): -1})
        reactions.append(r)
    
    print('Generate model')
    tm = ME_Model(id_or_model = 'tm', m_model = params.human_model)
    tm.add_reactions(reactions)
    tm.initialize_solver(solver_type='qminos', precision='quad')
    
    print('Solve')
    sln, stat, _ = tm.solve_lp(mu_val = mu_val)
    
    store = {'model': tm, 'sln': sln, 'stat': stat, 'sinks': metabolite_ids, 
            'infeasible_reactions': tm.infeasible_reactions(mu_val = mu_val, sln = sln, stat = stat)}
    return store

def par_sink(m_id_lists, n_cores):
    '''m_id_lists is a list of lists'''
    pool = multiprocessing.Pool(processes = n_cores)
    try:
        stores = pool.map(add_sink, m_id_lists)
        pool.close()
        pool.join()
        gc.collect()
        return stores
    except:
        pool.close()
        pool.join()
        gc.collect()
        raise ValueError('par failed')
    

In [8]:
# # res_ = {k:v for k,v in res_.items() if k == 0.052300000000000006 or k == 0.052250000000000005}
# mapper = dict(zip(sorted(res_.keys()), ['infeasible', 'feasible']))
# res_ = {mapper[k]:v for k,v in res_.items()}

with open('/data2/hratch/human_me/test_lp/' + 'infeasible_boundary.pickle', 'wb') as handle:
    pickle.dump(res[0.05225]['model'], handle) # boundary infeasible
#
 

# #expected status 1
# res[0.05225]['model'].initialize_solver(solver_type='qminos', precision='quad')
# test1 = res[0.05225]['model'].solve_lp(mu_val = 1e-9)

# with open('/data2/hratch/human_me/test_lp/' + 'infeasible_boundary.pickle', 'rb') as handle:
#     mod2 = pickle.load(handle) # boundary infeasible
# mod2.initialize_solver(solver_type='qminos', precision='quad')
# test2 = mod2.solve_lp(mu_val = 1e-9)

# test3 = add_sink(metabolite_ids = [])

In [21]:
r_ids = ['H2OGLYAQPt_R_0', 'GLYCTDle']
add = list()
for r_id in r_ids:
    add += [m.id for m in res[0.05225]['model'].reactions.get_by_id(r_id).metabolites]
add = sorted(set(add))

In [71]:
test = [[]] + [add] + [[m_id] for m_id in add] + [list(i) for i in list(itertools.combinations(add, 2))]
# first is negative control, second is positive control
stores = par_sink(m_id_lists = test,n_cores = len(test))

In [26]:
bog = [r.id for r in res[0.05225]['model'].boundary]
for store in stores:
    boundaries = [r.id for r in store['model'].boundary]
    if sorted([rid.replace('SK_', '') for rid in set(boundaries).difference(bog)]) != sorted(store['sinks']):
        raise ValueError('Something went wrong')

In [38]:
list(zip([store['sinks'] for store in stores], [store['stat'] for store in stores]))

[([], array(1)),
 (['HGNC:636_folded_protein_e', 'glyc_c', 'glyc_e', 'h2o_c', 'h2o_e'],
  array(0)),
 (['HGNC:636_folded_protein_e'], array(0)),
 (['glyc_c'], array(1)),
 (['glyc_e'], array(1)),
 (['h2o_c'], array(0)),
 (['h2o_e'], array(1)),
 (['HGNC:636_folded_protein_e', 'glyc_c'], array(0)),
 (['HGNC:636_folded_protein_e', 'glyc_e'], array(0)),
 (['HGNC:636_folded_protein_e', 'h2o_c'], array(0)),
 (['HGNC:636_folded_protein_e', 'h2o_e'], array(0)),
 (['glyc_c', 'glyc_e'], array(1)),
 (['glyc_c', 'h2o_c'], array(0)),
 (['glyc_c', 'h2o_e'], array(1)),
 (['glyc_e', 'h2o_c'], array(0)),
 (['glyc_e', 'h2o_e'], array(1)),
 (['h2o_c', 'h2o_e'], array(0))]

# explore more

In [75]:
res_ = {'infeasible': res[0.05225], 'feasible': stores[5]}

res_df = pd.DataFrame(columns = res_.keys(), index = [r.id for r in res_['infeasible']['model'].reactions])
for key in res_df.columns:
    for r_id in res_df.index:
        res_df.loc[r_id, key] = res_[key]['sln'][res_[key]['model'].reactions.index(r_id)]

res_df['difference'] = res_df.infeasible - res_df.feasible
# res_df = res_df[res_df.difference != 0]
res_df.drop(columns = ['difference'], inplace = True)

res_df['rel_diff'] =  (res_df.infeasible - res_df.feasible)/res_df.feasible
res_df['abs_rel_diff'] = ((res_df.infeasible - res_df.feasible)/res_df.feasible).abs()
res_df.sort_values(by = 'abs_rel_diff', ascending = False, inplace = True)

In [90]:
# r_ids = ['H2OGLYAQPt_R_0', 'H2OGLYAQPt_F_0', 'GLYCTDle']
# r_ids += [r.id for r in f_model.metabolites.get_by_id('h2o_c').reactions]

In [246]:
r_ids = ['H2OGLYAQPt_R_0', 'GLYCTDle', 'DHCRD2_0', 'RE2675C', 'RE3347C', 'LDH_L_F_0', 
        'r0782']
         #'DSAT', 'DHCRD1_0', 'r0245_F_0', 'LPS3', ''FADDP_0'']
# r_ids +=  [r.id for r in if_model.metabolites.get_by_id('h2o_c').reactions]

summ = res_df.loc[r_ids,:].sort_values(by = 'abs_rel_diff', ascending = False)
summ['r_id'] = summ.index
def temp(x):
    if x[1] < 0:
        return ';'.join([m.id for m in f_model.reactions.get_by_id(x[0]).reactants])
    else:
        return ';'.join([m.id for m in f_model.reactions.get_by_id(x[0]).products])
summ['prod'] = summ[['r_id', 'infeasible']].apply(lambda x: temp(x), axis = 1).tolist()
summ.drop(columns = ['r_id'], inplace = True)
summ

,infeasible,feasible,rel_diff,abs_rel_diff,blocked,prod
H2OGLYAQPt_R_0,1.60182e-07,0,inf,inf,0.000000e+00,h2o_e;glyc_e
DHCRD2_0,-6.64302e-10,0,-inf,inf,-6.644238e-10,fad_c;dhcrm_hs_c;HGNC:20113_folded_protein_c
r0782,8.48477e-30,4.30711e-47,1.96995e+17,1.96995e+17,7.912614e-32,gdpfuc_c;nad_c
RE3347C,7.45623e-10,8.13208e-11,8.16891,8.16891,7.457446e-10,nad_c;fadh2_c
GLYCTDle,1.17649e-07,-4.25323e-08,-3.76611,3.76611,-4.253234e-08,glyc_c
LDH_L_F_0,8.84872e-08,8.71586e-08,0.0152435,0.0152435,8.848746e-08,h_c;pyr_c;nadh_c
RE2675C,8.94697e-08,8.88054e-08,0.00748042,0.00748042,8.946987e-08,h2o_c;nad_c;crm_hs_c


In [90]:
# t = [i.split(';') for i in summ['prod'].values.tolist()]
# add = flat_list = [item for sublist in t for item in sublist]
# add = sorted(set(add).difference(['h2o_e', 'glyc_e', 'h2o_c', 'gyc_c']))

add = sorted(set(['fad_c', 'dhcrm_hs_c', 'HGNC:20113_folded_protein_c', 'gdpfuc_c', 'nad_c', 'fadh2_c', 
      'h_c', 'pyr_c', 'nadh_c', 'crm_hs_c']))
add = [[m_id] for m_id in add]
# add = [[m_id] for m_id in add]

In [58]:
stores2 = par_sink(m_id_lists = add,n_cores = len(add))

In [12]:
sinks = [store['sinks'][0] for store in stores2 if store['stat'].max() == 0]
sinks += ['h2o_c']

In [65]:
sinks

['HGNC:20113_folded_protein_c',
 'crm_hs_c',
 'dhcrm_hs_c',
 'gdpfuc_c',
 'h_c',
 'pyr_c',
 'h2o_c']

In [72]:
f_model = mod

In [101]:
r_ids = ['KHte', 'BHBt', 'EX_h_LPAREN_e_RPAREN_', 'EX_h_b']
         #'DSAT', 'DHCRD1_0', 'r0245_F_0', 'LPS3', ''FADDP_0'']
# r_ids = list()
# for m_id in ['crm_hs_c', 'dhcrm_hs_c', 'gdpfuc_c', 'pyr_c']:
#     r_ids += [r.id for r in mod.metabolites.get_by_id(m_id).reactions]
# r_ids.remove('SK_crm_hs_c')
summ = res_df.loc[r_ids,:].sort_values(by = 'abs_rel_diff', ascending = False)

summ['r_id'] = summ.index
def temp(x):
    if x[1] < 0:
        return ';'.join([m.id for m in f_model.reactions.get_by_id(x[0]).reactants])
    else:
        return ';'.join([m.id for m in f_model.reactions.get_by_id(x[0]).products])
summ['prod'] = summ[['r_id', 'infeasible']].apply(lambda x: temp(x), axis = 1).tolist()

summ.head(10)

,infeasible,feasible,rel_diff,abs_rel_diff,r_id,prod
BHBt,-2.21898e-07,-8.13199e-08,1.72871,1.72871,BHBt,h_e;bhb_e
KHte,0,-7.48988e-08,-1,1,KHte,h_e;k_c
EX_h_LPAREN_e_RPAREN_,1.07213e-07,6.64714e-06,-0.983871,0.983871,EX_h_LPAREN_e_RPAREN_,h_b
EX_h_b,1.07213e-07,6.64714e-06,-0.983871,0.983871,EX_h_b,


In [76]:
m_ids = [m.id for m in mod.metabolites]
for sink in sinks:
    if sink.replace('c', 'e') in m_ids:
        print(sink)

h_c
h2o_c


In [79]:
final_sinks = ['h_c', 'h2o_c']

In [83]:
for m_id in final_sinks:
    reactions = list(mod.metabolites.get_by_id(m_id).reactions)
    for r in reactions:
        if 'e' in r.compartments:
            if r.gene_reaction_rule == '':
                print(r.id)

RTOTALFATPc
KHte
r1088
BHBt


yo, i have a strange feasibility issue where if i had a sink for cytosolic hydrogen, model becomes feasible. but the various hydrogen transport reactions from boundary to cytosol don't require machinery and aren't bounded stringently. even if i go through those transport reactions and force flux through them to generate h_c, model is still infeasible. any thoughts?

In [88]:
mod.reactions.get_by_id('BHBt')

Reaction identifier,BHBt
Name,(R)-3-Hydroxybutanoate transport via H+ symport
Memory address,0x07febe8cf3240
Stoichiometry,bhb_e + h_e <=> bhb_c + h_c (R)-3-hydroxybutyrate + proton <=> (R)-3-hydroxybutyrate + proton
GPR,
Lower bound,-inf
Upper bound,inf


In [99]:
for m_id in ['h_b']:
    reactions = list(mod.metabolites.get_by_id(m_id).reactions)
    for r in reactions:
        if len(r.compartments) == 1:
            if r.gene_reaction_rule == '':
                print(r.id)

EX_h_b


In [ ]:
BHBt, KHte, EX_h_LPAREN_e_RPAREN_, EX_h_b

In [142]:
{'test_mod'} = [store['model'] for store in stores2 if store['sinks'] == ['h_c']][0]

In [143]:
test_mod.initialize_solver()
slnf,statf,_ = test_mod.solve_lp(mu_val = 1e-9)

Getting MINOS parameters...
Done in 239.809 seconds with status 0


In [154]:
with open('/data2/hratch/human_me/test_lp/' + 'infeasible_boundary.pickle', 'rb') as handle:
    fail = pickle.load(handle)

h_rxns = ['BHBt', 'EX_h_LPAREN_e_RPAREN_', 'EX_h_b']#, 'KHte']
reactions = [r.copy() for r in tqdm(fail.reactions) if r.id not in h_rxns]

r = fail.reactions.get_by_id('BHBt').copy()
r.lower_bound = -3e-7
reactions.append(r)

# r = fail.reactions.get_by_id('KHte').copy()
# r.lower_bound = 3e-7
# reactions.append(r)

r = fail.reactions.get_by_id('EX_h_LPAREN_e_RPAREN_').copy()
r.upper_bound = -3e-7
reactions.append(r)

r = fail.reactions.get_by_id('EX_h_b').copy()
r.upper_bound = -3e-7
reactions.append(r)

tm = ME_Model(id_or_model = 'tm', m_model = params.human_model)
tm.add_reactions(reactions)
tm.initialize_solver(solver_type='qminos', precision='quad')

print('Solve')
sln, stat, _ = tm.solve_lp(mu_val = 1e-9)

100%|██████████| 12650/12650 [02:22<00:00, 88.68it/s] 


Solve
Getting MINOS parameters...
Done in 215.671 seconds with status 1


In [155]:
res_ = {'infeasible': {'model': tm, 'sln': sln}, 'feasible': {'model': test_mod, 'sln': slnf}}

res_df = pd.DataFrame(columns = res_.keys(), index = [r.id for r in res_['infeasible']['model'].reactions])
for key in res_df.columns:
    for r_id in res_df.index:
        res_df.loc[r_id, key] = res_[key]['sln'][res_[key]['model'].reactions.index(r_id)]

res_df['difference'] = res_df.infeasible - res_df.feasible
# res_df = res_df[res_df.difference != 0]
res_df.drop(columns = ['difference'], inplace = True)

res_df['rel_diff'] =  (res_df.infeasible - res_df.feasible)/res_df.feasible
res_df['abs_rel_diff'] = ((res_df.infeasible - res_df.feasible)/res_df.feasible).abs()
res_df.sort_values(by = 'abs_rel_diff', ascending = False, inplace = True)



In [156]:
res_df.loc['SK_h_c', 'feasible'] = res_['feasible']['sln'][res_['feasible']['model'].reactions.index('SK_h_c')]

In [186]:
# r_ids = ['KHte', 'BHBt', 'EX_h_LPAREN_e_RPAREN_', 'EX_h_b', 'SK_h_c']
         #'DSAT', 'DHCRD1_0', 'r0245_F_0', 'LPS3', ''FADDP_0'']

r_ids = [r.id for r in res_['infeasible']['model'].metabolites.get_by_id('h_c').reactions if r.subsystem == '' and 'DECAPPING_mRNA_DEGRADATIONc' not in r.id]
r_ids.append('SK_h_c')
summ = res_df.loc[r_ids,:].sort_values(by = 'abs_rel_diff', ascending = False)

summ['r_id'] = summ.index
def prod(x):
    if x[1] < 0:
        return ';'.join([m.id for m in res_['feasible']['model'].reactions.get_by_id(x[0]).reactants])
    else:
        return ';'.join([m.id for m in res_['feasible']['model'].reactions.get_by_id(x[0]).products])

def react(x):
    if x[1] > 0:
        return ';'.join([m.id for m in res_['feasible']['model'].reactions.get_by_id(x[0]).reactants])
    else:
        return ';'.join([m.id for m in res_['feasible']['model'].reactions.get_by_id(x[0]).products])

summ['product'] = summ[['r_id', 'feasible']].apply(lambda x: prod(x), axis = 1).tolist()
summ['reactant'] = summ[['r_id', 'feasible']].apply(lambda x: react(x), axis = 1).tolist()


summ[summ.infeasible.abs() > 1e-19].head()

,infeasible,feasible,rel_diff,abs_rel_diff,r_id,product,reactant
ACACt2_R_0,2.12841e-07,0,inf,inf,ACACt2_R_0,h_e;acac_e,h_e;acac_e
PROt2r_R_0,1.94905e-07,0,inf,inf,PROt2r_R_0,h_e;pro_L_e,h_e;pro_L_e
GLYSNAT5tc_F,2.18534e-10,0,inf,inf,GLYSNAT5tc_F,h_e;na1_c;gly_c,h_e;na1_c;gly_c
BHBt,3e-07,8.71586e-08,2.442,2.442,BHBt,h_c;bhb_c,h_e;bhb_e
Htr,7.78227e-10,7.96626e-10,-0.0230953,0.0230953,Htr,h_r,h_c


In [189]:
res_['feasible']['model'].reactions.get_by_id('GLYSNAT5tc_F')

Reaction identifier,GLYSNAT5tc_F
Name,transport of Glycine into the cell coupled with co-transport with Sodium and counter transport wi...
Memory address,0x07fed0f341240
Stoichiometry,9.75579489046473e5*mu 1.95115897809295e6 HGNC:18070_folded_protein_e + gly_e + h_c + na1_e --> gly_c + h_e + na1_c 9.75579489046473e5*mu 1.95115897809295e6 + glycine + proton + sodium(1+) --> glycine + proton + sodium(1+)
GPR,HGNC:18070
Lower bound,0
Upper bound,inf


In [172]:
res_['feasible']['model'].reactions.get_by_id('SK_h_c')

Reaction identifier,SK_h_c
Name,
Memory address,0x07feaa3adc048
Stoichiometry,h_c <=> proton <=>
GPR,
Lower bound,-1000
Upper bound,1000


In [176]:
res_['feasible']['sln'][res_['feasible']['model'].reactions.index('SK_h_c')]

2.286153207108273e-07

I modified a coupling constraint and now model is infeasible unless I add this reaction: 'h_c <-> '. The feasible solution actually has forward flux, so 'h_c --> '. It's weirder cus there are a series of unbounded boundary reactions with no machinery that deliver h_c from outside...you seen anything like this? Why would it only be feasible if there is direct flux for cytosolic hydrogen to leave the model?

In [190]:
store = add_sink(metabolite_ids=['h_e'])

100%|██████████| 12650/12650 [01:59<00:00, 105.95it/s]


Generate model
Solve
Getting MINOS parameters...
Done in 257.073 seconds with status 1


In [194]:
boundary_metab = [m for m in params.human_model.metabolites if m.compartment == 'b']
test_m = []
for m_b in tqdm(boundary_metab):
    consider = False
    consider_2 = False
    
    reactions_b = m_b.reactions
    for r in reactions_b:
        if 'e' in r.compartments and len(r.genes) == 0:
            consider = True
    
    if consider:
        m_e = params.human_model.metabolites.get_by_id('_'.join(m_b.id.split('_'))[:-1] + 'e')
        reactions_e = m_e.reactions
        for r in reactions_e:
            if 'c' in r.compartments and len(r.genes) == 0:
                consider_2 = True 
    
    if consider_2:
        test_m.append(m_e.id)


100%|██████████| 84/84 [00:00<00:00, 25432.87it/s]


In [198]:
print(test_m)

['13_cis_retnglc_e', '3bcrn_e', '3ivcrn_e', 'HC00250_e', 'HC00342_e', 'ddca_e', 'Rtotal_e', 'acald_e', 'ahcys_e', 'asp_L_e', 'atp_e', 'bhb_e', 'c4crn_e', 'cit_e', 'crvnc_e', 'dcmp_e', 'glu_L_e', 'glyc_e', 'h2o2_e', 'h_e', 'o2_e', 'o2s_e', 'pe_hs_e', 'pglyc_hs_e', 'hdca_e', 'pro_L_e', 'ps_hs_e', 'retn_e', 'sbt_D_e', 'sph1p_e', 'utp_e', 'xmp_e']


In [7]:
test_m = sorted(set(['13_cis_retnglc_e', '3bcrn_e', '3ivcrn_e', 'HC00250_e', 'HC00342_e', 'ddca_e', 'Rtotal_e', 'acald_e', 
          'ahcys_e', 'asp_L_e', 'atp_e', 'bhb_e', 'c4crn_e', 'cit_e', 'crvnc_e', 'dcmp_e', 'glu_L_e', 'glyc_e', 
          'h2o2_e', 'h_e', 'o2_e', 'o2s_e', 'pe_hs_e', 'pglyc_hs_e', 'hdca_e', 'pro_L_e', 'ps_hs_e', 'retn_e', 
          'sbt_D_e', 'sph1p_e', 'utp_e', 'xmp_e']))
test_metab = [[m_id] for m_id in test_m]


In [ ]:
stores3 = par_sink(m_id_lists = test_m, n_cores = len(test_m))

Process ForkPoolWorker-42:
Process ForkPoolWorker-50:
Process ForkPoolWorker-63:
Process ForkPoolWorker-64:
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
  File "/usr/lib/python3.6/multiprocessing/process.py", line 258, in _bootstrap
    self.run()
  File "/usr/lib/python3.6/multiprocessing/process.py", line 258, in _bootstrap
    self.run()
Traceback (most recent call last):
  File "/usr/lib/python3.6/multiprocessing/process.py", line 258, in _bootstrap
    self.run()
  File "/usr/lib/python3.6/multiprocessing/process.py", line 93, in run
    self._target(*self._args, **self._kwargs)
  File "/usr/lib/python3.6/multiprocessing/process.py", line 258, in _bootstrap
    self.run()
  File "/usr/lib/python3.6/multiprocessing/process.py", line 93, in run
    self._target(*self._args, **self._kwargs)
  File "/usr/lib/python3.6/multiprocessing/process.py", line 93, in run
    self._target(*self._args, **self._kwargs)
  File "/usr/lib/p

Process ForkPoolWorker-55:
  File "/usr/lib/python3.6/multiprocessing/pool.py", line 44, in mapstar
    return list(map(*args))
  File "<ipython-input-2-c9adf99161a5>", line 8, in add_sink
    model = pickle.load(handle) # boundary infeasible
  File "/home/hratch/Projects/human_me/me_env/lib/python3.6/site-packages/sympy/core/numbers.py", line 1031, in __new__
    def __new__(cls, num, dps=None, prec=None, precision=None):
KeyboardInterrupt
Traceback (most recent call last):
  File "/usr/lib/python3.6/multiprocessing/process.py", line 258, in _bootstrap
    self.run()
  File "/usr/lib/python3.6/multiprocessing/process.py", line 93, in run
    self._target(*self._args, **self._kwargs)
  File "/usr/lib/python3.6/multiprocessing/pool.py", line 119, in worker
    result = (True, func(*args, **kwds))
  File "/usr/lib/python3.6/multiprocessing/pool.py", line 44, in mapstar
    return list(map(*args))
  File "<ipython-input-2-c9adf99161a5>", line 8, in add_sink
    model = pickle.load(handle)

KeyboardInterrupt
Process ForkPoolWorker-40:
Traceback (most recent call last):
  File "/usr/lib/python3.6/multiprocessing/process.py", line 258, in _bootstrap
    self.run()
  File "/usr/lib/python3.6/multiprocessing/process.py", line 93, in run
    self._target(*self._args, **self._kwargs)
  File "/usr/lib/python3.6/multiprocessing/pool.py", line 119, in worker
    result = (True, func(*args, **kwds))
  File "/usr/lib/python3.6/multiprocessing/pool.py", line 44, in mapstar
    return list(map(*args))
  File "<ipython-input-2-c9adf99161a5>", line 8, in add_sink
    model = pickle.load(handle) # boundary infeasible
  File "/home/hratch/Projects/human_me/me_env/lib/python3.6/site-packages/cobra/core/reaction.py", line 602, in __setstate__
    def __setstate__(self, state):
KeyboardInterrupt
Process ForkPoolWorker-59:
Traceback (most recent call last):
  File "/usr/lib/python3.6/multiprocessing/process.py", line 258, in _bootstrap
    self.run()
  File "/usr/lib/python3.6/multiprocessing

  File "<ipython-input-2-c9adf99161a5>", line 8, in add_sink
    model = pickle.load(handle) # boundary infeasible
  File "/home/hratch/Projects/human_me/me_env/lib/python3.6/site-packages/cobra/core/reaction.py", line 602, in __setstate__
    def __setstate__(self, state):
  File "/home/hratch/Projects/human_me/me_env/lib/python3.6/site-packages/sympy/core/numbers.py", line 1031, in __new__
    def __new__(cls, num, dps=None, prec=None, precision=None):
KeyboardInterrupt
KeyboardInterrupt
